# Cybershelf: from customer tables to product scores

This walkthrough uses synthetic data to explain ID alignment, model-specific features, and the 82-input neural stack. It does not reproduce competition accuracy.

In [ ]:
from pathlib import Path
import numpy as np
import torch
from cybershelf.demo import generate
from cybershelf.data import load_features, load_targets, submission
from cybershelf.features import FeatureBuilder
from cybershelf.stacking import StackingNet, focal_loss, predict_logits

torch.set_num_threads(2)
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
demo = root / "runs" / "walkthrough"
generate(demo)

## 1. Join by customer ID

The synthetic tables are shuffled independently. Features and labels are explicitly realigned; the identifier never becomes a model feature.

In [ ]:
x = load_features(demo / "main.parquet", demo / "extra.parquet")
y = load_targets(demo / "targets.parquet", x.customer_id)
assert len(x) == len(y) == 1000
assert y.shape[1] == 41
x.head()

## 2. Fit the frontend on training rows

Missingness and row statistics complement the original features. CatBoost receives strings; XGBoost receives a fixed pandas category dictionary.

In [ ]:
builder = FeatureBuilder().fit(x.iloc[:800])
catboost_inputs = builder.transform(x.iloc[800:], "catboost")
xgboost_inputs = builder.transform(x.iloc[800:], "xgboost")
assert catboost_inputs.shape == xgboost_inputs.shape
catboost_inputs.head()

## 3. Inspect the neural combination

Two sets of 41 held-out base probabilities give 82 inputs. In training, a QuantileTransformer fitted on meta-training data produces the input features. The following tensor is illustrative, not real tree predictions.

In [ ]:
torch.manual_seed(42)
model = StackingNet(input_dim=82, output_dim=41)
features = np.random.default_rng(42).normal(size=(7, 82)).astype("float32")
logits = predict_logits(model, features, batch_size=3)
assert logits.shape == (7, 41)
loss = focal_loss(torch.tensor(logits), torch.tensor(y.iloc[:7].to_numpy()))
assert torch.isfinite(loss)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))
print("Output:", logits.shape, "Illustrative focal loss:", loss.item())

## 4. Retain every customer and the target order

The submission contains logits for ranking, with one row per supplied ID. A sigmoid alone does not guarantee calibrated probabilities.

In [ ]:
result = submission(x.customer_id.iloc[:7], logits, y.columns.tolist())
assert result.shape == (7, 42)
assert result.customer_id.tolist() == x.customer_id.iloc[:7].tolist()
result.head()

## 5. Run the full pipeline

From the repository root, run the synthetic commands in README. Training writes the OOF table, customer split, settings, metrics and reloadable model bundle. Real-data experiments should use the documented outer-holdout protocol and dataset access rules.

## 6. Compare a standalone numerical baseline

The supplementary experiment uses numerical customer features directly, unlike the 82-input tree-prediction stacker. Its hidden layers are 512 and 256 units. This two-epoch synthetic example checks execution only.

In [ ]:
from cybershelf.neural_baseline import NumericBaseline

baseline = NumericBaseline().fit(x.iloc[:800], y.iloc[:800], epochs=2, batch_size=128)
baseline_logits = baseline.predict(x.iloc[800:], batch_size=73)
assert baseline_logits.shape == (200, 41)
print("Numerical inputs:", len(baseline.columns))
print("Training history:", baseline.history)